# Seeding Maintenance Events – Bronze → Silver

This notebook ingests Maintenance Events for welding robots into a medallion architecture:

- **Bronze layer** (`demo.bronze_layer.maintenance_events`): Raw maintenance event data.
- **Silver layer** (`demo.silver_layer.maintenance_events`): Cleansed copy ready for backend consumption.

**Source of truth:** `anomaly-detection/data/sql/seed_databricks_error_tables.sql` (20 rows, 4 robots, 5 events each).

**Requirements:** Databricks Runtime, SparkSession (`spark`).

In [ ]:
# %% Setup: Imports & Configuration
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Configuration — update catalog if your Unity Catalog differs
# ---------------------------------------------------------------------------
CATALOG         = "demo"
BRONZE_SCHEMA   = "bronze_layer"
SILVER_SCHEMA   = "silver_layer"
BRONZE_TABLE    = f"{CATALOG}.{BRONZE_SCHEMA}.maintenance_events"
SILVER_TABLE    = f"{CATALOG}.{SILVER_SCHEMA}.maintenance_events"

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Silver table: {SILVER_TABLE}")
print(f"Spark version: {spark.version}")

---
## Phase 1: Bronze Layer — Raw Maintenance Events Ingestion

In [ ]:
# %% Bronze: Create schema and table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    robot_code STRING,
    event_ts   TIMESTAMP,
    event_type STRING,
    notes      STRING
)
USING DELTA
""")

print(f"\u2713 Bronze table ensured: {BRONZE_TABLE}")
print(f"  Columns: robot_code, event_ts, event_type, notes")

In [ ]:
# %% Bronze: Seed 20 maintenance events (idempotent)
print("Clearing existing data for idempotent re-run...")
spark.sql(f"DELETE FROM {BRONZE_TABLE}")

print("Inserting 20 maintenance events...")

# ── R-00 (5 events) ────────────────────────────────────────────────────────
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('R-00', '2026-04-28 08:30:00', 'wire_feed_calibration',
 'Routine wire feed calibration performed. Drive roll pressure adjusted to 35 N. Feed rate verified at 8.5 m/min setpoint.'),
('R-00', '2026-05-02 14:15:00', 'contact_tip_replacement',
 'Contact tip replaced after 9.5 hours of arc-on time. Old tip bore measured 1.4 mm (nominal 1.2 mm).'),
('R-00', '2026-05-10 10:00:00', 'wire_temperature_sensor_check',
 'Wire temperature sensor verified against contact pyrometer. Offset +3\u00b0C corrected in control system.'),
('R-00', '2026-05-15 09:00:00', 'welding_current_clamp_inspection',
 'Welding current clamp inspected and cleaned. Ground connection resistance measured 3 mOhm \u2014 within spec.'),
('R-00', '2026-05-20 11:00:00', 'cable_drag_chain_inspection',
 'Cable drag chain inspected at R-00 wrist joint. One broken link found and replaced. Cable jacket intact.')
""")
print("  \u2713 R-00: 5 events inserted")

In [ ]:
# %% Bronze: Insert R-01 events (5 total)
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('R-01', '2026-04-29 07:45:00', 'wire_feed_calibration',
 'Wire feed system calibrated. Straightener tension reduced from 4.2 to 3.5. Feed roller pressure re-set to 32 N.'),
('R-01', '2026-05-05 13:30:00', 'contact_tip_replacement',
 'Scheduled contact tip replacement after 8 hours arc-on. Torch neck cleaned of spatter.'),
('R-01', '2026-05-12 09:15:00', 'cable_drag_chain_inspection',
 'Cable drag chain inspection at R-01. Found cable sheath wear at midpoint \u2014 protective wrap applied.'),
('R-01', '2026-05-18 10:30:00', 'welding_current_clamp_inspection',
 'Earth return cable connection inspected. Tightened at robot base. Resistance measured 4 mOhm.'),
('R-01', '2026-05-22 15:00:00', 'wire_temperature_sensor_check',
 'Coolant flow rate measured at 6.2 L/min (spec: 6.0 L/min). Temperature sensor reading within \u00b11\u00b0C of reference.')
""")
print("  \u2713 R-01: 5 events inserted")

In [ ]:
# %% Bronze: Insert R-02 events (5 total)
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('R-02', '2026-04-27 11:00:00', 'contact_tip_replacement',
 'Contact tip replaced. Noted excessive spatter on nozzle \u2014 cleaned and anti-spatter compound applied.'),
('R-02', '2026-05-03 08:00:00', 'wire_feed_calibration',
 'Feed motor bearing replaced due to audible noise. Feed calibration verified after repair. Setpoint tracking within \u00b11%.'),
('R-02', '2026-05-11 14:00:00', 'cable_drag_chain_inspection',
 'Full drag chain inspection at R-02. No damage found. All links moving freely.'),
('R-02', '2026-05-17 10:00:00', 'welding_current_clamp_inspection',
 'Secondary cable connections at power source checked and torqued. No oxidation found.'),
('R-02', '2026-05-21 12:30:00', 'wire_temperature_sensor_check',
 'Chiller unit inspected. Radiator fins cleaned with compressed air. Coolant level topped up.')
""")
print("  \u2713 R-02: 5 events inserted")

In [ ]:
# %% Bronze: Insert R-03 events (5 total)
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('R-03', '2026-04-26 09:00:00', 'wire_feed_calibration',
 'Full wire feed path cleaned. Debris removed from inlet guide. Drive rolls inspected \u2014 minor wear noted, scheduled for replacement.'),
('R-03', '2026-05-04 15:30:00', 'contact_tip_replacement',
 'Contact tip and gas nozzle replaced. Gas flow rate verified at 18 L/min.'),
('R-03', '2026-05-09 08:45:00', 'welding_current_clamp_inspection',
 'Primary power supply phase balance checked. Imbalance 3.2% \u2014 within acceptable limits.'),
('R-03', '2026-05-16 11:00:00', 'cable_drag_chain_inspection',
 'Cable drag chain inspected at R-03. One cable tie broken \u2014 replaced. All cables in good condition.'),
('R-03', '2026-05-23 10:00:00', 'wire_temperature_sensor_check',
 'Coolant hose routing adjusted at robot elbow to prevent kinking. Temperature sensor cable shielding repaired at connector.')
""")
print("  \u2713 R-03: 5 events inserted")

In [ ]:
# %% Bronze: Verification
bronze_check = spark.sql(f"""
SELECT robot_code, event_type, COUNT(*) AS event_count
FROM {BRONZE_TABLE}
GROUP BY robot_code, event_type
ORDER BY robot_code, event_type
""")

total_rows = bronze_check.select(F.sum("event_count")).collect()[0][0]
print("=" * 60)
print("BRONZE LAYER \u2014 MAINTENANCE EVENTS VERIFICATION")
print("=" * 60)
bronze_check.show(truncate=False)
print(f"\nTotal rows: {total_rows} \u2705")
assert total_rows == 20, f"Expected 20, got {total_rows}"

---
## Phase 2: Silver Layer \u2014 Cleaned Events for Backend Consumption

In [ ]:
# %% Silver: Create schema and table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    robot_code STRING,
    event_ts   TIMESTAMP,
    event_type STRING,
    notes      STRING
)
USING DELTA
""")

print(f"\u2713 Silver table ensured: {SILVER_TABLE}")
print(f"  Same schema as bronze \u2014 cleansed copy for backend queries")

In [ ]:
# %% Silver: Copy from bronze with overwrite
bronze_df = spark.table(BRONZE_TABLE)

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"\u2713 Silver table written: {SILVER_TABLE}")

In [ ]:
# %% Silver: Verification
silver_check = spark.sql(f"""
SELECT robot_code, COUNT(*) AS event_count
FROM {SILVER_TABLE}
GROUP BY robot_code
ORDER BY robot_code
""")

silver_count = spark.table(SILVER_TABLE).count()
print("\n" + "=" * 60)
print("SILVER LAYER \u2014 VERIFICATION")
print("=" * 60)
print(f"Total rows: {silver_count}")
print(f"\nEvents per robot:")
silver_check.show(truncate=False)

assert silver_count == 20, f"Expected 20 rows, got {silver_count}"
print(f"\n\u2705 Silver layer seeded successfully \u2014 {silver_count} maintenance events")

---
## Summary

### Created Resources

| Layer  | Resource | Location |
|--------|----------|----------|
| Bronze | Delta table (20 raw rows) | `demo.bronze_layer.maintenance_events` |
| Silver | Delta table (20 rows) | `demo.silver_layer.maintenance_events` |

### Maintenance Event Types (5 per robot)

| Event Type | Description |
|------------|-------------|
| `wire_feed_calibration` | Wire feed system calibration / drive roll adjustment |
| `contact_tip_replacement` | Contact tip (and nozzle) replacement |
| `wire_temperature_sensor_check` | Temperature sensor / coolant system check |
| `welding_current_clamp_inspection` | Electrical connections / ground resistance check |
| `cable_drag_chain_inspection` | Cable drag chain / wrist joint inspection |

### Table Schema

| Column | Type | Description |
|--------|------|-------------|
| `robot_code` | STRING | Robot identifier (R-00 to R-03) |
| `event_ts` | TIMESTAMP | When the maintenance was performed |
| `event_type` | STRING | Category of maintenance activity |
| `notes` | STRING | Detailed description of the work done |

**Source of truth:** `anomaly-detection/data/sql/seed_databricks_error_tables.sql`